# News Coverage Classification Model

This notebook trains and evaluates a news coverage classification model.

## Model Architecture
- **Feature Extraction**: TF-IDF Vectorizer
- **Classifier**: Linear SVC
- **Task**: Multi-class topic classification

In [1]:
# Imports
import pandas as pd
import numpy as np
import re, csv, warnings, joblib
from collections import Counter
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

print('✓ All imports loaded!')

✓ All imports loaded!


## 1. Data Loading

In [2]:
COLS = ['id','label','statement','subjects','speaker','job_title',
        'state_info','party_affiliation','barely_true_cnt','false_cnt',
        'half_true_cnt','mostly_true_cnt','pants_on_fire_cnt','context','justification']

def read_tsv(path):
    return pd.read_csv(path, sep='\t', header=None, names=COLS, engine='python',
                       quoting=csv.QUOTE_NONE, escapechar='\\', on_bad_lines='skip')

def text_of(r):
    return ' '.join([str(r.get('statement','')), str(r.get('context','')), str(r.get('justification',''))]).strip()

def first_subject(s):
    parts = re.split(r'[;,]', s) if isinstance(s,str) else []
    return parts[0].strip().lower() if parts and parts[0].strip() else 'unknown'

print('✓ Functions defined!')

✓ Functions defined!


In [4]:
# Load datasets
df_tr = read_tsv('data/train2.tsv')
df_va = read_tsv('data/valid.tsv')
df_te = read_tsv('data/test2.tsv')

X_tr, X_va, X_te = df_tr.apply(text_of, axis=1), df_va.apply(text_of, axis=1), df_te.apply(text_of, axis=1)
y_tr, y_va, y_te = df_tr['subjects'].apply(first_subject), df_va['subjects'].apply(first_subject), df_te['subjects'].apply(first_subject)

keep = y_tr.ne('unknown')
X_tr, y_tr = X_tr[keep], y_tr[keep]

print(f'Train: {len(X_tr)}, Val: {len(X_va)}, Test: {len(X_te)}')
print(f'Unique topics: {y_tr.nunique()}')

Train: 10269, Val: 1284, Test: 1299
Unique topics: 140


## 2. Model Training

In [5]:
# Compute class weights
classes = np.unique(y_tr)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_tr)
wmap = {c:w for c,w in zip(classes, weights)}

print(f'✓ Class weights computed for {len(wmap)} classes')

✓ Class weights computed for 140 classes


In [6]:
# Train model
news_coverage_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, strip_accents='unicode', analyzer='word',
                              ngram_range=(1,2), min_df=2, max_df=0.9, sublinear_tf=True)),
    ('clf', LinearSVC(class_weight=wmap, random_state=42, max_iter=2000))
])

news_coverage_pipe.fit(X_tr, y_tr)
print('\n✓ Model training completed!')


✓ Model training completed!


## 3. Model Evaluation

In [7]:
# Predictions
y_pred_tr = news_coverage_pipe.predict(X_tr)
y_pred_va = news_coverage_pipe.predict(X_va)
y_pred_te = news_coverage_pipe.predict(X_te)

acc_tr = accuracy_score(y_tr, y_pred_tr)
acc_va = accuracy_score(y_va, y_pred_va)
acc_te = accuracy_score(y_te, y_pred_te)

print('='*60)
print('ACCURACY SCORES')
print('='*60)
print(f'Training:   {acc_tr:.4f} ({acc_tr*100:.2f}%)')
print(f'Validation: {acc_va:.4f} ({acc_va*100:.2f}%)')
print(f'Test:       {acc_te:.4f} ({acc_te*100:.2f}%)')

ACCURACY SCORES
Training:   0.9945 (99.45%)
Validation: 0.3279 (32.79%)
Test:       0.2548 (25.48%)


In [8]:
# Macro metrics
print('\n'+'='*60)
print('MACRO-AVERAGED METRICS')
print('='*60)

for split_name, y_true, y_pred in [('Training', y_tr, y_pred_tr), ('Validation', y_va, y_pred_va), ('Test', y_te, y_pred_te)]:
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    print(f'\n{split_name}:')
    print(f'  Precision: {p:.4f}')
    print(f'  Recall:    {r:.4f}')
    print(f'  F1-Score:  {f1:.4f}')


MACRO-AVERAGED METRICS

Training:
  Precision: 0.9869
  Recall:    0.9993
  F1-Score:  0.9928

Validation:
  Precision: 0.1763
  Recall:    0.2373
  F1-Score:  0.1725

Test:
  Precision: 0.1474
  Recall:    0.1898
  F1-Score:  0.1526


In [9]:
# Classification report
print('\n'+'='*60)
print('CLASSIFICATION REPORT (Validation)')
print('='*60)
print(classification_report(y_va, y_pred_va, zero_division=0))


CLASSIFICATION REPORT (Validation)
                         precision    recall  f1-score   support

               abortion       0.78      0.64      0.71        28
            afghanistan       0.20      0.50      0.29         2
            agriculture       0.17      0.20      0.18         5
                alcohol       0.00      0.00      0.00         3
                animals       0.00      0.00      0.00         5
             bankruptcy       0.00      0.00      0.00         6
               baseball       0.00      0.00      0.00         1
         bipartisanship       0.33      0.29      0.31        17
    bush-administration       0.00      0.00      0.00         5
   campaign-advertising       0.00      0.00      0.00         4
       campaign-finance       0.44      0.31      0.36        26
   candidates-biography       0.50      0.15      0.24        65
          cap-and-trade       0.29      0.50      0.36         4
                 census       0.00      0.00      0.0

In [10]:
# Per-class metrics
p, r, f1, support = precision_recall_fscore_support(y_va, y_pred_va, average=None, zero_division=0, labels=classes)
metrics_df = pd.DataFrame({'Topic': classes, 'Support': support, 'Precision': p, 'Recall': r, 'F1-Score': f1})
metrics_df = metrics_df.sort_values('Support', ascending=False)

print('\nTop 20 Classes by Support:')
print(metrics_df.head(20).to_string(index=False))


Top 20 Classes by Support:
               Topic  Support  Precision   Recall  F1-Score
             economy      116   0.708333 0.146552  0.242857
         health-care      113   0.727273 0.566372  0.636816
           education       81   0.779661 0.567901  0.657143
candidates-biography       65   0.500000 0.153846  0.235294
      federal-budget       53   0.409091 0.169811  0.240000
      foreign-policy       45   0.454545 0.222222  0.298507
         immigration       41   0.625000 0.609756  0.617284
           elections       40   0.423077 0.275000  0.333333
               crime       32   0.318182 0.218750  0.259259
              energy       31   0.586207 0.548387  0.566667
               taxes       30   0.428571 0.500000  0.461538
            abortion       28   0.782609 0.642857  0.705882
    campaign-finance       26   0.444444 0.307692  0.363636
        state-budget       26   0.636364 0.269231  0.378378
                jobs       23   0.187500 0.391304  0.253521
            